# 01 — Scraping Ulasan Access by KAI

Mengambil ulasan pengguna aplikasi **Access by KAI** (`com.kai.kaiticketing`) dari
Google Play Store menggunakan pustaka `google-play-scraper`.

- Bahasa: `id`, Negara: `id`
- Hasil disimpan ke `data/raw/reviews_access_by_kai.csv`
- Data mentah **tidak boleh diedit langsung**

In [1]:
from pathlib import Path
from datetime import datetime

import pandas as pd
from google_play_scraper import Sort, reviews

ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)

APP_ID = "com.kai.kaiticketing"
TARGET = 5000

print("Root proyek :", ROOT)
print("Folder data :", DATA_RAW)
print("Aplikasi    :", APP_ID)
print("Target ulasan:", TARGET)

Root proyek : C:\skripsi
Folder data : C:\skripsi\data\raw
Aplikasi    : com.kai.kaiticketing
Target ulasan: 5000


In [2]:
result, _ = reviews(
    APP_ID,
    lang="id",
    country="id",
    sort=Sort.NEWEST,
    count=TARGET,
)

df = pd.DataFrame(result)
print("Jumlah ulasan terambil:", len(df))
print("Kolom:", list(df.columns))

Jumlah ulasan terambil: 5000
Kolom: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']


In [3]:
keep = [
    "reviewId", "userName", "content", "score", "thumbsUpCount",
    "reviewCreatedVersion", "at", "replyContent", "repliedAt",
]
keep = [c for c in keep if c in df.columns]
df = df[keep].copy()
df["scraped_at"] = datetime.now().isoformat(timespec="seconds")

print("Baris:", len(df))
print("Ulasan tanpa teks:", (df["content"].fillna("").str.strip() == "").sum())
print("\nDistribusi bintang:")
print(df["score"].value_counts().sort_index())
df.head()

Baris: 5000
Ulasan tanpa teks: 0

Distribusi bintang:
score
1    1673
2     302
3     319
4     291
5    2415
Name: count, dtype: int64


,reviewId,userName,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,scraped_at
0,3af55b39-23a3-4d9c-9dc4-4be509450ade,Pengguna Google,kalo ga mau duit lu ilang mending jangan downl...,1,0,NaN,2026-10-08 22:50:15,NaN,NaT,2026-10-10T01:26:05
1,e50c861d-ade7-47c7-957e-d8450f8cbb6e,Pengguna Google,ketika cancel tiket veritifikasi kode otp suda...,1,0,NaN,2026-10-08 22:20:15,NaN,NaT,2026-10-10T01:26:05
2,f56c333e-7191-49fe-8e0f-983bc0c83fbc,Pengguna Google,mantappp,5,0,6.19.6,2026-10-08 21:41:15,NaN,NaT,2026-10-10T01:26:05
3,6fad3c00-4b5e-45a2-b0f4-837c0e7d652a,Pengguna Google,"terlalu sering up -grade2,capek,aktivisasi kai...",1,0,6.19.6,2026-10-08 21:14:31,NaN,NaT,2026-10-10T01:26:05
4,4f3a5083-3b35-4589-a94b-0cccf13cf039,Pengguna Google,"Mohon untuk di perbaharui di nomer hp, agar bi...",3,0,6.19.6,2026-10-08 20:56:44,NaN,NaT,2026-10-10T01:26:05


In [4]:
out_path = DATA_RAW / "reviews_access_by_kai.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Tersimpan ke:", out_path)
print("Ukuran berkas:", out_path.stat().st_size, "byte")

Tersimpan ke: C:\skripsi\data\raw\reviews_access_by_kai.csv
Ukuran berkas: 910673 byte
